# 02 - Création des variables cibles

On part de (`data/processed/comptes_rendus_nettoyes.csv`) , et on construit deux variables cibles :

- **SentimentLabel** : Positive / Neutral / Negative
- **ChurnRisk** : score de risque de départ combinant 4 signaux, puis classement Élevé / Moyen / Faible

Approche retenue : **règles à base de mots-clés**

### 1. Chargement de la data nettoyée

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import ast
import os
import numpy as np
import pandas as pd
from google.colab import files

# Sélection manuelle du fichier
uploaded = files.upload()

# Récupérer le nom du fichier sélectionné
nom_fichier = next(iter(uploaded))

# Charger le CSV
df = pd.read_csv(nom_fichier)

# Les Tokens sont stockés comme chaînes dans le CSV
df["Tokens"] = df["Tokens"].apply(ast.literal_eval)

print("Corpus chargé :", nom_fichier)
print("Dimensions :", df.shape)

display(df.head(5))

Saving comptes_rendus_nettoyes_1000.csv to comptes_rendus_nettoyes_1000.csv
Corpus chargé : comptes_rendus_nettoyes_1000.csv
Dimensions : (1000, 13)


,ID_Client,Date_Contact,Canal,Compte_Rendu_Text,Code_Motif,text_length,word_count,has_arabic,has_latin,Langue,Tokens,Compte_Rendu_Clean,word_count_clean
0,C_0001,2026-06-17,Application,Client de passage au guichet pour contester de...,MOTIF_FRAIS,193,31,False,True,Français,"[client, passage, guichet, pour, contester, ag...",client passage guichet pour contester agios il...,24
1,C_0002,2026-05-09,Agence,الحريف يشتكي من قلة احترام العون في فرع تونس. ...,MOTIF_RELA_CONSEIL,113,22,True,False,TunAr,"[حريف, اشتكي, من, قله, احترام, عون, في, فرع, ت...",حريف اشتكي من قله احترام عون في فرع تونس قال د...,23
2,C_0003,2026-05-28,Application,Le client signale un dysfonctionnement bloquan...,MOTIF_DIGITAL,153,18,False,True,Français,"[client, signale, dysfonctionnement, bloquant,...",client signale dysfonctionnement bloquant lors...,16
3,C_0004,2026-05-14,Téléphone,Client de passage au guichet pour contester de...,MOTIF_FRAIS,193,31,False,True,Français,"[client, passage, guichet, pour, contester, ag...",client passage guichet pour contester agios il...,24
4,C_0005,2026-05-13,Téléphone,Le client signale un dysfonctionnement bloquan...,MOTIF_DIGITAL,153,18,False,True,Français,"[client, signale, dysfonctionnement, bloquant,...",client signale dysfonctionnement bloquant lors...,16


### 2.2 - SentimentLabel

| Label | Interprétation |
|---|---|
| Positive | Satisfaction, remerciement, expérience favorable |
| Neutral | Demande / information sans polarité claire |
| Negative | Plainte, insatisfaction, colère, problème |

In [3]:
# Lexiques de mots-clés

import re

import re

MOTS_CLES_POSITIFS = {
    # français
    "merci", "remercier", "remerciement", "satisfait", "satisfaction", "content",
    "apprécier", "parfait", "excellent", "agréable", "bravo","merci", "rapide", "efficace",
                 "parfait", "rapidite", "remercie","rapidité","rapide",
    # arabizi / tunisien (latin)
    "behi", "mizyen", "chokran", "yezzik", "saha", "far7an","farhan","far7anin","farhanin","behi","mezyena","mezyen","mabrouk","3aychek","3ayechkom","sa7a","hamdoulah", "hamdoullah", "behi", "saha", "sahit", "chokrane",
                "mrigel", "yaatik","mriguel",
    # arabe (formes attendues après lemmatisation CAMeL Tools)
    "شكر", "راض", "ممتاز", "جيد", "سعيد", "مبسوط","شكرا", "ممتاز", "راضي", "الحمد","باهية","يشكر","ممتازة","راضية","الحمد لله","باهية","يشكرون","ممتازين","راضين","الحمد لله","باهيين",
}

MOTS_CLES_NEGATIFS = {
    # français — mots de plainte explicites
    "plainte", "insatisfait", "insatisfaction", "colère", "problème", "mécontent",
    "déçu", "inacceptable", "scandaleux", "erreur", "refuser", "rejeter", "bloquer",
    "incident", "dysfonctionnement", "réclamation", "réclamer", "échec","bug","bugs","catastrophe",
    # français — problème non résolu / en attente (souvent sans mot "négatif" explicite)
    "conservé", "avalé", "avalée", "sans réponse", "sans suite", "resté sans",
    "restée sans", "en attente", "toujours pas", "toujours en attente",
    "aucune réponse", "aucune suite", "non traité", "non traitée", "non résolu",
    "non résolue", "manquant", "manquante", "retard", "n'a pas été", "n'arrive pas",
    "pas reçu", "pas encore reçu", "pas remboursé","refusé","manque",
    # arabizi / tunisien — plainte explicite
    "mochkel", "khayeb","khayba","5ayeb","5ayba", "3ayen", "ghaltan", "zeft", "ma3jebnich","fdhi7a","fdhiha",
    # arabizi / tunisien — problème non résolu / négation dialectale (suffixe "-ch/-ech")
    "ma3adech", "mazal", "mayebanch", "mawselch", "matsajjlech", "matwasslouch",
    "manjamch", "matefadhalch","ta3tel","t3atel","ma t5demch","mate5demch", "matekhdemch","ma wesletch","mawesletch",
    # arabe — plainte explicite
    "مشكل", "غلط", "سيء", "رفض", "معطل", "شكو", "غاضب", "ضايق", "اشكال",
    # arabe — problème non résolu / négation dialectale (suffixe "ش")
    "وصلوش", "ماعادش", "ماتسجلش", "بقاش", "تأخير", "معلق", "ماوصلوش", "ماينجمش","ما عادش","ما وصلش ",
}

# "merci de ..." est une formule de politesse administrative ("merci de bien vouloir
# répondre"), pas un signe de gratitude réelle -> ne doit pas déclencher "Positive".

_MERCI_FORMEL_RE = re.compile(r"merci\s+de\b")

# Cues de négation qui inversent le sens d'un mot négatif juste après
_NEGATION_RE = re.compile(
    r"\b(sans|aucun[e]?|pas de|plus de|plus aucun[e]?)\s+(\w+\s+){0,2}"
)

def compter_occurrences(texte, lexique):
    """Compte le nombre de mots-clés distincts trouvés (pas juste un booléen)."""
    return sum(1 for mot in lexique if re.search(r"\b" + re.escape(mot) + r"\b", texte))

def compter_negatifs_hors_negation(texte, lexique_neg):
    """Compte les mots négatifs, en excluant ceux précédés d'une négation
    (ex. 'sans problème', 'aucun souci')."""
    n = 0
    for mot in lexique_neg:
        for m in re.finditer(re.escape(mot), texte):
            debut = max(0, m.start() - 20)
            contexte_avant = texte[debut:m.start()]
            if not _NEGATION_RE.search(contexte_avant + " "):
                n += 1
    return n

def classifier_sentiment(texte):
    texte = str(texte).lower()

    score_neg = compter_negatifs_hors_negation(texte, MOTS_CLES_NEGATIFS)

    autres_mots_positifs = MOTS_CLES_POSITIFS - {"merci"}
    score_pos = compter_occurrences(texte, autres_mots_positifs)
    if ("merci" in texte) and not _MERCI_FORMEL_RE.search(texte):
        score_pos += 1

    if score_neg > score_pos:
        return "Negative"
    elif score_pos > score_neg:
        return "Positive"
    else:
        return "Neutral"  # égalité ou aucun mot-clé -> neutre


df["SentimentLabel"] = df["Compte_Rendu_Clean"].apply(classifier_sentiment)
df["SentimentLabel"].value_counts()

,count
SentimentLabel,
Neutral,623
Negative,329
Positive,48


### 2. Configuration

`LANG_COL` doit pointer vers la colonne réellement présente dans le DataFrame (`Langue`), avec ses vraies valeurs
(`Français`, `TunLat`, `TunAr`, `Mixte`
`CONF_THRESHOLD` : en dessous de ce score de confiance du modèle, on laisse une chance au filet lexical de
trancher (utile pour les cas ambigus / dialectal mal couvert par le modèle).

In [4]:
CONFIG = {
    "MODEL_NAME": "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual",
    "TEXT_COL": "Compte_Rendu_Clean",
    "LANG_COL": "Langue",
    "CONF_THRESHOLD": 0.75,
    "BATCH_SIZE": 32,
}

assert CONFIG["TEXT_COL"] in df.columns, f"{CONFIG['TEXT_COL']} absent du DataFrame"
assert CONFIG["LANG_COL"] in df.columns, f"{CONFIG['LANG_COL']} absent du DataFrame"

### 3. Filet lexical

In [5]:

import re

LEXIQUE_NEGATIF_TOKENS = {
    "problème", "probleme", "dysfonctionnement", "bloque", "bloqué", "refus",
    "insatisfait", "deçu", "déçu", "scandaleux", "inadmissible", "frustrant",
    "nul", "erreur", "réclamation", "incident", "mochkel", "mochkla", "za3fan",
    "5ayeb", "khayeb", "ma3andich", "mamnou3", "matzabetch",
    "tarde", "tarder", "mazal", "ta3tel", "ta3adet",
}

# Racines arabes cherchées en SOUS-CHAÎNE (pas en token exact) : le dialectal
# tunisien colle souvent un préfixe verbal sans espace ("تعطلو", "ترفض"),
# donc une comparaison de tokens exacts les rate systématiquement.
NEG_SUBSTR_ARABE = ["مشكلة", "مشكل", "اشكال", "إشكال", "غاضب", "رفض", "معطل", "تعطل"]

LEXIQUE_POSITIF_TOKENS = {
    "merci", "satisfait", "satisfaction", "content", "bravo", "excellent",
    "agréable", "remercie", "remerciement", "apprécie",
    "hamdoulah", "hamdoullah", "behi", "saha", "sahit", "chokrane", "chokran",
    "mizyen", "farhan", "mabrouk",
    "شكرا", "ممتاز", "راضي", "الحمد", "شكر", "سعيد", "مبسوط",
}

# IMPORTANT : ces tournures doivent être cherchées dans le texte BRUT
# (Compte_Rendu_Text), pas dans Compte_Rendu_Clean -- le nettoyage de la
# phase 1 retire les mots-outils (en, sans, à, de, un...) qui composent
# une bonne partie de ces expressions ("en retard" -> "retard" après
# nettoyage, ce qui casse le matching de phrase).
NEG_PHRASES = [
    "n'a pas", "ne fonctionne plus", "ne répond pas", "sans réponse", "sans explication",
    "sans solution", "reste en attente", "reste vide", "n'arrive pas", "pas de réponse",
    "pas encore", "toujours pas", "n'est pas arrivé", "non reçu", "non visible",
    "ne correspond pas", "trop long", "trop lente", "trop élevé", "trop élevés",
    "trop faible", "jugé", "jugée", "jugées", "excessif", "excessifs", "insuffisant",
    "insuffisante", "incorrecte", "incorrect", "refusé", "refusée", "bloqué", "bloquée",
    "en retard", "tardive", "tardif", "tardivement", "annulation", "rejet", "rejeté",
    "conteste", "considéré comme", "considérée comme", "n'a pas reçu", "sans retour",
    "sans remise complète", "malgré", "plusieurs tentatives", "en attente", "qui tarde",
    "ma l9a", "l9ach", "9ach", "ma ynajjem","t9os","bug","bugs","rzina",
]

NEG_SUFFIX_LATIN = re.compile(r"\b[a-z0-9]{2,}ch\b")
NEG_SUFFIX_ARABIC = re.compile(r"[\u0600-\u06FF]{2,}ش\b")


def score_lexical(text, langue=None):
    """Retourne 'Positif' / 'Négatif' / None. À appeler sur le texte BRUT."""
    t = str(text).lower()
    toks = set(re.findall(r"\w+", t))

    a_signal_negatif = bool(
        toks & LEXIQUE_NEGATIF_TOKENS
        or any(p in t for p in NEG_PHRASES)
        or any(s in text for s in NEG_SUBSTR_ARABE)
        or NEG_SUFFIX_LATIN.search(t)
        or NEG_SUFFIX_ARABIC.search(text)
    )
    a_signal_positif = bool(toks & LEXIQUE_POSITIF_TOKENS)

    if a_signal_negatif and not a_signal_positif:
        return "Négatif"
    if a_signal_positif and not a_signal_negatif:
        return "Positif"
    if a_signal_positif and a_signal_negatif:
        return "Négatif"
    return None

### 4. Modèle zero-shot (signal principal)

`cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual` couvre le français et l'arabe standard ; sa
couverture du dialectal tunisien / arabizi est imparfaite mais reste le meilleur choix disponible en zero-shot
sans fine-tuning.

In [6]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Device utilisé :", "GPU" if device == 0 else "CPU")

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=CONFIG["MODEL_NAME"],
    truncation=True,
    device=device,
)

LABEL_MAP = {"positive": "Positif", "neutral": "Neutre", "negative": "Négatif"}


def label_zero_shot(textes, batch_size=32):
    resultats = []
    n = len(textes)
    for i in range(0, n, batch_size):
        batch = [str(t) for t in textes[i:i + batch_size]]
        preds = sentiment_pipeline(batch)
        for p in preds:
            resultats.append((LABEL_MAP.get(p["label"].lower(), "Neutre"), p["score"]))
        if (i // batch_size) % 10 == 0:
            print(f"  {min(i + batch_size, n)}/{n} lignes traitées")
    return resultats


Device utilisé : GPU


config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [7]:
resultats = label_zero_shot(df[CONFIG["TEXT_COL"]].tolist(), batch_size=CONFIG["BATCH_SIZE"])
df["sentiment_modele"] = [r[0] for r in resultats]
df["confiance_modele"] = [r[1] for r in resultats]

df[[CONFIG["TEXT_COL"], "sentiment_modele", "confiance_modele"]].head(10)

  32/1000 lignes traitées


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  352/1000 lignes traitées
  672/1000 lignes traitées
  992/1000 lignes traitées


,Compte_Rendu_Clean,sentiment_modele,confiance_modele
0,client passage guichet pour contester agios il...,Négatif,0.931726
1,حريف اشتكي من قله احترام عون في فرع تونس قال د...,Négatif,0.977200
2,client signale dysfonctionnement bloquant lors...,Négatif,0.955169
3,client passage guichet pour contester agios il...,Négatif,0.931726
4,client signale dysfonctionnement bloquant lors...,Négatif,0.955169
5,حريف جا فرع غاضب علي خاطر تقصتلو خط متعه شي بد...,Négatif,0.897052
6,rendez vous suivi annuel constructif client se...,Positif,0.637560
7,reçu mail متاع client y7eb ya3mel rdv معا cons...,Neutre,0.832197
8,مطلب قرض استهلاك شراء سيار مستعمل ورق كامل ملف...,Neutre,0.961231
9,appel client y7eb ya3mel virement compte compt...,Neutre,0.652570


### 5. Fusion modèle + filet lexical

In [8]:
def fusion_label(row):
    lex = score_lexical(row["Compte_Rendu_Text"])  # texte BRUT, pas Compte_Rendu_Clean
    modele = row["sentiment_modele"]
    conf = row["confiance_modele"]

    if modele == "Positif":
        if lex == "Positif":
            return "Positif"
        if lex == "Négatif":
            return "Négatif"
        return "Neutre"

    if lex == "Négatif":
        return "Négatif"
    if lex == "Positif":
        return "Positif"

    if conf >= CONFIG["CONF_THRESHOLD"]:
        return modele
    return "Neutre"


df["SentimentLabel"] = df.apply(fusion_label, axis=1)
df["SentimentLabel_source"] = "silver"

print(df["SentimentLabel"].value_counts())
print(df["SentimentLabel"].value_counts(normalize=True).round(3))

SentimentLabel
Négatif    881
Neutre      71
Positif     48
Name: count, dtype: int64
SentimentLabel
Négatif    0.881
Neutre     0.071
Positif    0.048
Name: proportion, dtype: float64


### 6. Contrôle qualité manuel


In [9]:
pd.set_option("display.max_colwidth", None)

for label in ["Positif", "Neutre", "Négatif"]:
    sous_ens = df[df["SentimentLabel"] == label]
    print(f"\n{'='*70}\n{label}  (n={len(sous_ens)})\n{'='*70}")
    if len(sous_ens) == 0:
        print("  (aucune ligne)")
        continue
    echantillon = sous_ens.sample(min(5, len(sous_ens)), random_state=42)
    for _, r in echantillon.iterrows():
        print(f"- [{r['confiance_modele']:.2f}] {r['Compte_Rendu_Text']}")



Positif  (n=48)
- [0.64] Rendez-vous de suivi annuel constructif. Le client se dit pleinement satisfait des performances de son assurance-vie et de la réactivité de son conseiller.
- [0.64] Rendez-vous de suivi annuel constructif. Le client se dit pleinement satisfait des performances de son assurance-vie et de la réactivité de son conseiller.
- [0.64] Rendez-vous de suivi annuel constructif. Le client se dit pleinement satisfait des performances de son assurance-vie et de la réactivité de son conseiller.
- [0.64] Rendez-vous de suivi annuel constructif. Le client se dit pleinement satisfait des performances de son assurance-vie et de la réactivité de son conseiller.
- [0.64] Rendez-vous de suivi annuel constructif. Le client se dit pleinement satisfait des performances de son assurance-vie et de la réactivité de son conseiller.

Neutre  (n=71)
- [0.95] Demande d'informations concernant les conditions d'octroi d'un crédit immobilier (taux d'intérêt, durée maximale, autofinancement req

## 2.3 - ChurnRisk

Le risque est construit à partir de 4 signaux :

| Signal | Poids | Définition |
|---|---|---|
| Intention de départ | 40% | Mots-clés indiquant une volonté de quitter la banque |
| Sentiment négatif | 25% | SentimentLabel = Négatif |
| Récurrence | 20% | Même client + même motif ≥ 2 contacts |
| Sensibilité du motif | 15% | Poids attribué au motif bancaire |

**Formule :**
`Score = 0,40 × Départ + 0,25 × Négatif + 0,20 × Récurrence + 0,15 × Sensibilité`

**Classement :** appliqué uniquement aux comptes rendus **Négatifs** (seuls les clients mécontents sont de vrais candidats au churn). Les seuils (quantiles 0,33 et 0,66) sont calculés sur ce sous-ensemble Négatif, puis chaque compte rendu négatif est classé Élevé / Moyen / Faible. Les comptes rendus Positive/Neutral reçoivent la valeur `"Non applicable"`.

In [10]:
# --- Signal 1 : Intention de départ (mots-clés, 40%) ---

def contient_un_mot_cle(texte, lexique):
    """Recherche par sous-chaîne : couvre aussi bien les mots seuls que les
    expressions à plusieurs mots (ex. 'fermer mon compte')."""
    return any(mot in str(texte).lower() for mot in lexique)


MOTS_CLES_DEPART = {
    # français
    "fermer mon compte", "fermer le compte", "clôturer", "résilier", "quitter la banque",
    "changer de banque", "autre banque", "dernière fois", "plus jamais", "plus confiance",
    "plus confiance en cette banque",
    # arabizi / tunisien
    "nsaker", "nbeddel", "nbadel", "banka okhra", "banka o5ra", "bank ekher",
    "akher marra", "ma3adich net3amel", "ekher mara", "e5er mara",
    # arabe
    "نغلق حساب", "نبدل بنك", "بنك آخر", "آخر مرة",
}


def a_intention_de_depart(texte):
    return int(contient_un_mot_cle(texte, MOTS_CLES_DEPART))


df["Signal_Depart"] = df["Compte_Rendu_Text"].apply(a_intention_de_depart)
df["Signal_Depart"].value_counts()

,count
Signal_Depart,
0,901
1,99


In [11]:
# --- Signal 2 : Sentiment négatif (25%) ---
df["Signal_Negatif"] = (df["SentimentLabel"] == "Négatif").astype(int)
df["Signal_Negatif"].value_counts()

,count
Signal_Negatif,
1,881
0,119


In [12]:
# --- Signal 3 : Récurrence (20%) ---
# Même client + même motif >= 2 contacts
compte_par_client_motif = df.groupby(["ID_Client", "Code_Motif"])["Code_Motif"].transform("count")
df["Signal_Recurrence"] = (compte_par_client_motif >= 2).astype(int)
df["Signal_Recurrence"].value_counts()

,count
Signal_Recurrence,
0,1000


In [13]:
# --- Signal 4 : Sensibilité du motif (15%) ---
# Poids business (0-1) par motif bancaire : plus le motif touche directement
# à l'argent du client (frais, crédit, retrait, virement), plus la sensibilité est élevée.

POIDS_MOTIF = {
    "MOTIF_FRAIS": 0.90,
    "MOTIF_CREDIT": 0.90,
    "MOTIF_RETRAIT": 0.85,
    "MOTIF_VIREMENT": 0.80,
    "MOTIF_COMPTE": 0.75,
    "MOTIF_CARTE": 0.70,
    "MOTIF_CHEQUE": 0.55,
    "MOTIF_DIGITAL": 0.50,
    "MOTIF_RELATION": 0.45,
    "MOTIF_CONSEILLER": 0.40,
     "MOTIF_CLOTURE": 0.95,
    "MOTIF_MONETIQUE": 0.70,
    "MOTIF_RELA_CONSEIL": 0.45,
}

motifs_inconnus = set(df["Code_Motif"].unique()) - set(POIDS_MOTIF.keys())
if motifs_inconnus:
    print(" Motifs sans poids défini (poids par défaut 0.5 appliqué) :", motifs_inconnus)

df["Signal_Sensibilite"] = df["Code_Motif"].map(POIDS_MOTIF).fillna(0.5)
df["Signal_Sensibilite"].value_counts()


,count
Signal_Sensibilite,
0.90,410
0.50,215
0.70,185
0.45,96
0.95,94


In [14]:
print(df.shape)
print(df.columns.tolist())
print(df["Langue"].value_counts())

(1000, 21)
['ID_Client', 'Date_Contact', 'Canal', 'Compte_Rendu_Text', 'Code_Motif', 'text_length', 'word_count', 'has_arabic', 'has_latin', 'Langue', 'Tokens', 'Compte_Rendu_Clean', 'word_count_clean', 'SentimentLabel', 'sentiment_modele', 'confiance_modele', 'SentimentLabel_source', 'Signal_Depart', 'Signal_Negatif', 'Signal_Recurrence', 'Signal_Sensibilite']
Langue
TunLat      380
TunAr       302
Français    231
Mixte        87
Name: count, dtype: int64


In [15]:
# --- Score combiné ---
# Calculé pour toutes les lignes (les 4 signaux restent informatifs quel que soit le sentiment).
df["ChurnScore"] = (
    0.40 * df["Signal_Depart"]
    + 0.25 * df["Signal_Negatif"]
    + 0.20 * df["Signal_Recurrence"]
    + 0.15 * df["Signal_Sensibilite"]
)

# --- Classement en terciles : uniquement sur les comptes rendus Négatifs ---
# Seuls les clients mécontents sont de vrais candidats au churn ; on calcule
# donc les seuils (0.33 / 0.66) sur ce sous-ensemble, pas sur tout le corpus
# (sinon la masse de scores nuls des Positive/Neutral fausserait les seuils).
masque_negatif = df["SentimentLabel"] == "Négatif"
scores_negatifs = df.loc[masque_negatif, "ChurnScore"]

assert masque_negatif.sum() > 0, "Aucune ligne Négatif trouvée — vérifie SentimentLabel.unique()"

seuil_bas, seuil_haut = np.quantile(scores_negatifs, [0.33, 0.66])
print(f"Seuils (terciles, calculés sur les {masque_negatif.sum()} comptes rendus Négatifs) : {seuil_bas:.3f} / {seuil_haut:.3f}")

def classer_churn(score):
    if score <= seuil_bas:
        return "Faible"
    elif score <= seuil_haut:
        return "Moyen"
    else:
        return "Élevé"

df["ChurnRisk"] = "Non applicable"
df.loc[masque_negatif, "ChurnRisk"] = df.loc[masque_negatif, "ChurnScore"].apply(classer_churn)
df["ChurnRisk"].value_counts()

Seuils (terciles, calculés sur les 881 comptes rendus Négatifs) : 0.355 / 0.385


,count
ChurnRisk,
Faible,448
Moyen,240
Élevé,193
Non applicable,119


In [16]:
# Vérification sur l'exemple du brief : Départ=1, Négatif=1, Récurrence=0, Motif=0.6
score_exemple = 0.40 * 1 + 0.25 * 1 + 0.20 * 0 + 0.15 * 0.6
print("Score attendu (exemple du brief) :", round(score_exemple, 2))  # doit être 0.74

Score attendu (exemple du brief) : 0.74


In [17]:
# Aperçu final
df[[
    "ID_Client", "Code_Motif", "SentimentLabel",
    "Signal_Depart", "Signal_Negatif", "Signal_Recurrence", "Signal_Sensibilite",
    "ChurnScore", "ChurnRisk"
]].sample(10, random_state=42)

,ID_Client,Code_Motif,SentimentLabel,Signal_Depart,Signal_Negatif,Signal_Recurrence,Signal_Sensibilite,ChurnScore,ChurnRisk
521,C_0522,MOTIF_FRAIS,Négatif,0,1,0,0.90,0.3850,Moyen
737,C_0738,MOTIF_DIGITAL,Négatif,0,1,0,0.50,0.3250,Faible
740,C_0741,MOTIF_CREDIT,Négatif,0,1,0,0.90,0.3850,Moyen
660,C_0661,MOTIF_RELA_CONSEIL,Négatif,0,1,0,0.45,0.3175,Faible
411,C_0412,MOTIF_RELA_CONSEIL,Négatif,0,1,0,0.45,0.3175,Faible
678,C_0679,MOTIF_CREDIT,Négatif,0,1,0,0.90,0.3850,Moyen
626,C_0627,MOTIF_MONETIQUE,Négatif,0,1,0,0.70,0.3550,Faible
513,C_0514,MOTIF_CREDIT,Neutre,0,0,0,0.90,0.1350,Non applicable
859,C_0860,MOTIF_CLOTURE,Négatif,0,1,0,0.95,0.3925,Élevé
136,C_0137,MOTIF_FRAIS,Négatif,0,1,0,0.90,0.3850,Moyen


## 3. Sauvegarde

In [18]:
from pathlib import Path

dossier_data = Path("/content/data")
dossier_sortie = dossier_data / "processed"

dossier_sortie.mkdir(parents=True, exist_ok=True)

chemin_fichier = dossier_sortie / "comptes_rendus_variables_cibles.csv"

df.to_csv(chemin_fichier, index=False)

print("Fichier sauvegardé :", chemin_fichier.resolve())

Fichier sauvegardé : /content/data/processed/comptes_rendus_variables_cibles.csv
